In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df= pd.read_csv('/kaggle/input/snappfood-persian-sentiment-analysis/Snappfood - Sentiment Analysis.csv',encoding='utf-8', sep='\t+', on_bad_lines='skip')

df = df.head(3000)
df = df[['comment', 'label']]
df = df.dropna()

<ipython-input-1-9998a693ffb8>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df= pd.read_csv('/kaggle/input/snappfood-persian-sentiment-analysis/Snappfood - Sentiment Analysis.csv',encoding='utf-8', sep='\t+', on_bad_lines='skip')


In [2]:
!pip install hazm

In [3]:
from hazm import Normalizer

# نرمال‌سازی متن‌ها
normalizer = Normalizer()

def preprocess_text(text):
    # حذف فاصله‌های اضافی، تبدیل اعداد انگلیسی به فارسی، و نرمال‌سازی متن
    return normalizer.normalize(text)

# اعمال پیش‌پردازش روی ستون 'comment'
df['comment'] = df['comment'].apply(preprocess_text)

# بررسی نتیجه
print(df.head())

# ذخیره دیتاست پیش‌پردازش شده
# data.to_csv("Snappfood-Sentiment Analysis_preprocessed.csv", index=False)
# print("پیش‌پردازش انجام شد و دیتاست ذخیره گردید.")

                                             comment  label
0    واقعا حیف وقت که بنویسم سرویس دهیتون شده افتضاح    SAD
1  قرار بود ۱ ساعته برسه ولی نیم ساعت زودتر از مو...  HAPPY
2  قیمت این مدل اصلا با کیفیتش سازگاری نداره، فقط...    SAD
3  عالی بود همه چه درست و به اندازه و کیفیت خوب، ...  HAPPY
4                      شیرینی وانیلی فقط یک مدل بود.  HAPPY


In [4]:
def map_labels(rate):
    if rate == "HAPPY":
        return 1
    else:
        return 0


df['label'] = df['label'].apply(map_labels)

In [5]:
# تقسیم داده‌ها به سه مجموعه (آموزش 50%، اعتبارسنجی 20%، تست 30%)
train_data, temp_data = train_test_split(df, test_size=0.5, stratify=df['label'], random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.6, stratify=temp_data['label'], random_state=42)


print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

# ذخیره داده‌ها برای استفاده‌های بعدی
train_data.to_csv("snappfood_train.csv", index=False)
val_data.to_csv("snappfood_val.csv", index=False)
test_data.to_csv("snappfood_test.csv", index=False)

print("دیتاست آماده شد و ذخیره گردید.")

Training samples: 1500
Validation samples: 600
Test samples: 900
دیتاست آماده شد و ذخیره گردید.


In [6]:
!pip install transformers

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import torch
import pandas as pd

In [8]:
# بارگذاری مدل و توکنایزر ParsBERT
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased-sentiment-snappfood"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# تعریف کلاس دیتاست برای PyTorch
class snappfoodDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]

        # توکنایز کردن متن
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
# بارگذاری داده‌های تقسیم‌شده
train_data = pd.read_csv("snappfood_train.csv")
val_data = pd.read_csv("snappfood_val.csv")
test_data = pd.read_csv("snappfood_test.csv")

In [10]:
# تعریف دیتاست‌ها
train_dataset = snappfoodDataset(
    texts=train_data["comment"].tolist(),
    labels=train_data["label"].tolist(),
    tokenizer=tokenizer,
    max_len=128,
)

val_dataset = snappfoodDataset(
    texts=val_data["comment"].tolist(),
    labels=val_data["label"].tolist(),
    tokenizer=tokenizer,
    max_len=128,
)

test_dataset = snappfoodDataset(
    texts=test_data["comment"].tolist(),
    labels=test_data["label"].tolist(),
    tokenizer=tokenizer,
    max_len=128,
)

In [11]:
# تعریف DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("مدل و داده‌ها آماده شدند.")

مدل و داده‌ها آماده شدند.


In [12]:
# تا اینجا رو از چت موزیلا گرفتم
# بقیه رو خودم زدم

In [13]:
import torch
import time
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# انتقال مدل به GPU در صورت وجود
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# تنظیمات بهینه‌ساز و Scheduler
optimizer = AdamW(model.parameters(), lr=5e-5)
scheduler = StepLR(optimizer, step_size=3, gamma=0.7)
criterion = torch.nn.CrossEntropyLoss()

num_epochs = 14

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    # شروع زمان اپوک
    start_time = time.time()

    # آموزش مدل
    model.train()
    epoch_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # ذخیره پیش‌بینی‌ها و برچسب‌ها برای ارزیابی
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    # بروزرسانی Learning Rate
    scheduler.step()

    # محاسبه معیارهای ارزیابی
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    # زمان پایان اپوک
    end_time = time.time()
    epoch_time = end_time - start_time

    # چاپ نتایج اپوک
    print(f"Epoch {epoch + 1} Results:")
    print(f"  Training Loss: {epoch_loss / len(train_loader):.4f}")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"  Epoch Time: {epoch_time:.2f} seconds")

    # ارزیابی مدل روی داده‌های اعتبارسنجی (Validation)
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(labels.cpu().numpy())

    # محاسبه معیارهای اعتبارسنجی
    val_accuracy = accuracy_score(val_labels, val_preds)
    val_precision = precision_score(val_labels, val_preds, average='weighted')
    val_recall = recall_score(val_labels, val_preds, average='weighted')
    val_f1 = f1_score(val_labels, val_preds, average='weighted')

    print(f"Validation Results:")
    print(f"  Validation Loss: {val_loss / len(val_loader):.4f}")
    print(f"  Validation Accuracy: {val_accuracy:.4f}")
    print(f"  Validation Precision: {val_precision:.4f}")
    print(f"  Validation Recall: {val_recall:.4f}")
    print(f"  Validation F1-Score: {val_f1:.4f}")

print("آموزش مدل به پایان رسید.")


Epoch 1/14


100%|██████████| 94/94 [00:19<00:00,  4.80it/s]


Epoch 1 Results:
  Training Loss: 0.4642
  Accuracy: 0.8007
  Precision: 0.8080
  Recall: 0.8007
  F1-Score: 0.7995
  Learning Rate: 0.000050
  Epoch Time: 19.61 seconds


100%|██████████| 38/38 [00:02<00:00, 17.21it/s]


Validation Results:
  Validation Loss: 0.2972
  Validation Accuracy: 0.8750
  Validation Precision: 0.8891
  Validation Recall: 0.8750
  Validation F1-Score: 0.8739
Epoch 2/14


...

Validation Results:
  Validation Loss: 0.7462
  Validation Accuracy: 0.8467
  Validation Precision: 0.8482
  Validation Recall: 0.8467
  Validation F1-Score: 0.8465
Epoch 14/14


100%|██████████| 94/94 [00:19<00:00,  4.87it/s]


Epoch 14 Results:
  Training Loss: 0.0013
  Accuracy: 0.9993
  Precision: 0.9993
  Recall: 0.9993
  F1-Score: 0.9993
  Learning Rate: 0.000012
  Epoch Time: 19.32 seconds


100%|██████████| 38/38 [00:02<00:00, 17.30it/s]

Validation Results:
  Validation Loss: 0.7595
  Validation Accuracy: 0.8500
  Validation Precision: 0.8516
  Validation Recall: 0.8500
  Validation F1-Score: 0.8498
آموزش مدل به پایان رسید.


In [14]:
# ذخیره مدل و توکنایزر
model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")

print("مدل و توکنایزر با موفقیت ذخیره شدند.")

مدل و توکنایزر با موفقیت ذخیره شدند.


In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# تابع ارزیابی مدل روی داده‌های تست
def evaluate_model(model, dataloader, device):
    model.eval()  # تغییر وضعیت مدل به ارزیابی
    all_preds = []
    all_labels = []

    with torch.no_grad():  # غیرفعال کردن محاسبات گرادیان
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1).cpu().numpy()  # پیش‌بینی‌ها
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())  # برچسب‌ها

    # محاسبه معیارها
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    # نمایش نتایج
    print("Evaluation Results:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")

# ارزیابی مدل روی داده‌های تست
evaluate_model(model, test_loader, device)

100%|██████████| 57/57 [00:03<00:00, 17.34it/s]

Evaluation Results:
  Accuracy: 0.8822
  Precision: 0.8835
  Recall: 0.8822
  F1-Score: 0.8821


In [16]:
'''
from transformers import AdamW
from torch.optim import lr_scheduler
import torch.nn as nn
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import time
'''

'\nfrom transformers import AdamW\nfrom torch.optim import lr_scheduler\nimport torch.nn as nn\nfrom tqdm import tqdm\nfrom sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score\nimport time\n'

In [17]:
'''

# تنظیمات اولیه
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# تعریف معیار و بهینه‌ساز
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=5e-5)

# تنظیم زمانبند یادگیری
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# تعداد ایپاک‌ها
num_epochs = 3

# حلقه آموزش
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    start_time = time.time()

    model.train()
    epoch_loss = 0
    all_preds = []
    all_labels = []
    
    # آموزش در هر Batch
    for batch in tqdm(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        
        # ذخیره پیش‌بینی‌ها و برچسب‌ها
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    # به‌روزرسانی زمانبند یادگیری
    scheduler.step()

    # محاسبه معیارهای اعتبارسنجی
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    # زمان اجرا
    end_time = time.time()
    epoch_time = end_time - start_time

    # چاپ نتایج
    print(f"Epoch {epoch + 1} Results:")
    print(f"  Training Loss: {epoch_loss / len(dataloader):.4f}")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"  Epoch Time: {epoch_time:.2f} seconds")

print("آموزش مدل به پایان رسید.")
'''


'\n\n# تنظیمات اولیه\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nmodel = model.to(device)\n\n# تعریف معیار و بهینه\u200cساز\ncriterion = nn.CrossEntropyLoss()\noptimizer = AdamW(model.parameters(), lr=5e-5)\n\n# تنظیم زمانبند یادگیری\nscheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)\n\n# تعداد ایپاک\u200cها\nnum_epochs = 3\n\n# حلقه آموزش\nfor epoch in range(num_epochs):\n    print(f"Epoch {epoch + 1}/{num_epochs}")\n\n    start_time = time.time()\n\n    model.train()\n    epoch_loss = 0\n    all_preds = []\n    all_labels = []\n    \n    # آموزش در هر Batch\n    for batch in tqdm(dataloader):\n        input_ids = batch["input_ids"].to(device)\n        attention_mask = batch["attention_mask"].to(device)\n        labels = batch["labels"].to(device)\n\n        optimizer.zero_grad()\n        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)\n        loss = outputs.loss\n        logits = outputs.logits\n\n        loss.bac